<a href="https://colab.research.google.com/github/pradh/tools/blob/ml/UncuratedStatVarEmbeddings_3000.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Prepare Data by Querying BQ

Gets StatVar Name-ID pairs for about 3.5K variables (up to 3 PVs), spanning:
- Demographics: gender, race, poverty, education, disability, benefits, marital
- Family
- Housing
- Farms
- Emissions
- Crime (could be helpful to uncover embarassing results)
- Health: outcome, prevention, behavior

This just uses the auto-gen names now, and should be improved, a lot.

**TODO: expand the name strings using LLMs**

In [1]:
%load_ext google.colab.data_table

import os
os.environ["GOOGLE_CLOUD_PROJECT"] = 'datcom-store'

from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
client = bigquery.Client()

NAME_AND_SV = client.query("""
SELECT name, id
FROM `datcom-store.dc_kg_latest.StatisticalVariable`
WHERE
   name IS NOT NULL AND
   id NOT LIKE '%/%' AND
   population_type IN ('Person', 'Household', 'HousingUnit', 'CriminalActivities', 'Emissions', 'FarmInventory') AND
   num_constraints < 4 AND
   (num_constraints = 0 OR
     (num_constraints > 0 AND
      p1 IN ('medicalCondition', 'gender', 'race', 'povertyStatus', 'educationalAttainment', 'crimeType',
             'incomeStatus', 'healthBehavior', 'healthOutcome', 'healthPrevention', 'benefitsStatus', 'farmInventoryType',
             'disabilityStatus', 'healthInsurance', 'maritalStatus', 'emissionSource', 'emittedThing')) OR
     (num_constraints > 1 AND
      p1 <> 'age' AND
      p2 IN ('medicalCondition', 'gender', 'race', 'povertyStatus', 'educationalAttainment', 'crimeType',
            'incomeStatus', 'healthBehavior', 'healthOutcome', 'healthPrevention', 'benefitsStatus', 'farmInventoryType',
            'disabilityStatus', 'healthInsurance', 'maritalStatus', 'emissionSource', 'emittedThing')))
ORDER BY id;
""").to_dataframe()

NAME_AND_SV

,name,id
0,Amount of Farm Inventory: Winter Wheat for Grain,AmountFarmInventory_WinterWheatForGrain
1,CO2 Emissions Per Capita,Amount_Emissions_CarbonDioxide_PerCapita
2,Amount of Farm Inventory: Barley for Grain,Amount_FarmInventory_BarleyForGrain
3,Amount of Farm Inventory: Corn for Silage or G...,Amount_FarmInventory_CornForSilageOrGreenchop
4,Amount of Farm Inventory: Cotton,Amount_FarmInventory_Cotton
...,...,...
3432,Unemployment Rate: Male,UnemploymentRate_Person_Male
3433,"Unemployment Rate: Female, Rural",UnemploymentRate_Person_Rural_Female
3434,"Unemployment Rate: Male, Rural",UnemploymentRate_Person_Rural_Male
3435,"Unemployment Rate: Female, Urban",UnemploymentRate_Person_Urban_Female


In [2]:
len(NAME_AND_SV['id'].values.tolist())

3437

## 2. Build embeddings

Uses a pre-trained model to build embeddings for the above SV strings.

Must run this before you can search.  Downloading the model (~80MB) takes a minute maybe and embedding building is another minute or two.

TODO: try the larger / better(?) multi-QA model ([see this](https://www.sbert.net/docs/pretrained_models.html#semantic-search)).

In [3]:
%%capture
!pip install -U sentence-transformers
!pip install datasets

In [4]:
from sentence_transformers import SentenceTransformer, util

# Download model
model = SentenceTransformer('all-MiniLM-L6-v2')

texts = NAME_AND_SV['name'].values.tolist()
dcids = NAME_AND_SV['id'].values.tolist()

embeddings = model.encode(texts)

import pandas as pd
embeddings = pd.DataFrame(embeddings)
embeddings

Downloading:   0%|          | 0.00/1.18k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/190 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/10.6k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/612 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

Downloading:   0%|          | 0.00/350 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/13.2k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/232k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/349 [00:00<?, ?B/s]

,0,1,2,3,4,5,6,7,8,9,...,374,375,376,377,378,379,380,381,382,383
0,-0.012348,0.008449,-0.009229,0.035883,0.014810,0.013421,-0.057414,-0.009755,-0.034140,0.009522,...,0.029015,0.015599,-0.026001,-0.092131,0.047983,-0.046486,-0.046950,-0.102830,-0.064731,-0.028013
1,0.088758,-0.016812,0.029042,-0.005452,0.069181,-0.016399,0.019746,0.025571,-0.014354,0.026204,...,0.072013,0.009652,-0.028526,0.049000,-0.037699,-0.040841,0.038144,0.031133,-0.004361,-0.052929
2,0.020381,-0.002699,-0.001624,0.014707,0.019020,0.001249,-0.044839,-0.018978,-0.036714,0.001769,...,0.062622,-0.008461,-0.015896,-0.109937,0.039300,-0.049487,-0.012520,-0.104593,-0.015283,-0.098316
3,0.009556,-0.008747,0.006300,-0.027788,0.005100,-0.019342,-0.043632,0.022153,-0.025405,-0.008202,...,0.033413,-0.043064,-0.000999,-0.080541,0.060753,-0.079178,0.005522,-0.089621,0.037027,-0.002745
4,-0.029800,-0.017785,-0.069694,0.018113,0.017854,0.014047,-0.030409,0.003266,-0.031698,0.045925,...,0.033709,-0.058789,-0.060910,-0.064629,0.084547,-0.008348,-0.092361,-0.106022,0.003193,-0.067278
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3432,-0.000985,0.054984,0.017761,0.054060,0.001579,0.004966,-0.027473,-0.066635,-0.083630,-0.007368,...,0.096502,-0.101940,-0.055506,-0.006550,-0.110486,-0.011417,0.095437,-0.081324,-0.003027,-0.016828
3433,0.008582,-0.013613,0.033892,0.099452,0.023769,0.063728,-0.081211,-0.035717,-0.061377,0.048094,...,0.078641,-0.084849,0.021649,-0.067605,-0.055341,-0.006471,-0.017555,-0.069531,0.033066,-0.025557
3434,0.025818,0.019072,0.029659,0.079555,0.019502,0.033894,-0.061688,-0.058087,-0.086991,0.021953,...,0.110090,-0.105496,0.014389,-0.030015,-0.057145,-0.045049,0.010485,-0.077093,0.011161,-0.021823
3435,0.047191,-0.019447,0.060001,0.094334,-0.008146,0.063102,-0.068345,-0.044701,-0.089046,0.037309,...,0.084838,-0.111980,-0.041362,-0.054100,-0.089721,0.001862,0.054178,-0.054568,0.011799,-0.005621


In [5]:
import torch
from datasets import load_dataset

# Mimic save / load of embeddings.  (Really only necessary when index becomes bigger)

embeddings.to_csv("embeddings.csv", index=False)
ds = load_dataset('csv', data_files='embeddings.csv')
dataset_embeddings = torch.from_numpy(ds["train"].to_pandas().to_numpy()).to(torch.float)

Extracting data files:   0%|          | 0/1 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Dataset csv downloaded and prepared to /root/.cache/huggingface/datasets/csv/default-0c9b369ed522ae8d/0.0.0/6b34fb8fcf56f7c8ba51dc895bfa2bfbe43546f190a60fcf74bb5e8afdcc2317. Subsequent calls will reuse this data.


  0%|          | 0/1 [00:00<?, ?it/s]

## 3. Search

Given a search query, uses the model downloaded above (so run the above cells first!) to compute query embeddings, and calls the [`semantic_search`](https://www.sbert.net/examples/applications/semantic-search/README.html#util-semantic-search) function.  Per docs, the function computes exact nearest-neighbor using Cosine similarity match by default.

*Run the cell below once, and then auto-completion will work (sometimes needs a space at the end)*

TODO: Understand score values, especially for results unrelated to the query.

In [6]:
#@title { run: "auto", vertical-output: true }
QUERY = "food production" #@param {type:"string"}

query_embeddings = model.encode([QUERY])

from sentence_transformers.util import semantic_search
hits = semantic_search(query_embeddings, dataset_embeddings, top_k=10)

svs = [dcids[e['corpus_id']] for e in hits[0]]
scores = [e['score'] for e in hits[0]]
result = pd.DataFrame({'SV': svs, 'Cosine Score': scores})

result

,SV,Cosine Score
0,Count_FarmInventory_Broilers,0.505568
1,Amount_FarmInventory_DryEdibleBeans,0.490002
2,Count_Person_Producer_TwoOrMoreRaces,0.474581
3,Amout_FarmInventory_CornForGrain,0.456537
4,Annual_Emissions_GreenhouseGas_FoodProcessingB...,0.451107
5,Amount_FarmInventory_Rice,0.447723
6,Amount_FarmInventory_CornForSilageOrGreenchop,0.446847
7,Count_FarmInventory_MilkCows,0.432999
8,Amount_FarmInventory_WheatForGrain,0.429032
9,Amount_FarmInventory_SugarbeetsForSugar,0.426208
